In [ ]:
import sys

import pandas as pd
import nibabel as nib
import numpy as np
import scipy
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, losses
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard, ReduceLROnPlateau

import os
import datetime

from pathlib import Path

from tensorflow.keras.callbacks import CSVLogger

import random
from tensorflow.keras.optimizers import Adam



AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
def generate_visit_pairs(path_to_csv):
    df = pd.read_csv(path_to_csv, delimiter=";")

    df['CDR'] = df['CDR'].astype(str).str.replace(',', '.')
    df['CDR'] = pd.to_numeric(df['CDR'], errors='coerce')

    # Ensure consistent sorting
    df = df.sort_values(by=['Subject ID', 'Visit'])

    pairs = []

    for id, group in df.groupby('Subject ID'):
        visits = group.sort_values(by='Visit').reset_index(drop=True)

        for i in range(len(visits) - 1):
            v0 = visits.iloc[i]
            v1 = visits.iloc[i + 1]

            cdr0 = v0['CDR']
            cdr1 = v1['CDR']

            # Create progression label
            # CDR rose
            if cdr1 > cdr0:
                label = 2
            # CDR stable
            elif cdr1 == cdr0:
                label = 1
            # CDR reverted
            elif cdr1 < cdr0:
                label = 0
            
            pairs.append((v0['MRI ID'], v1['MRI ID'], label))

    return pairs

def resize_and_load_img(filename: str):
    img = nib.load("../Images/" + filename + "/RAW/mpr-1.nifti.hdr")
    img_data = img.get_fdata()

    img_data = ((img_data - np.mean(img_data)) / np.std(img_data)).astype(np.float32)

    """
    plt.imshow(data[data.shape[0] // 2], cmap="gray")
    plt.show()
    """

    img_data = img_data[..., 0]

    zoom_factors = [t / c for t, c in zip((128, 128, 128, 1), img_data.shape)]
    volume = scipy.ndimage.zoom(img_data, zoom_factors, order=1)
    
    volume = volume[..., np.newaxis]
    
    return volume

pairs = generate_visit_pairs("./oasis_longitudinal_demographics.csv") # LaterDO: files from cmd

"""
scans = []
labels = []


for scan1, scan2, label in pairs:

    img1 = resize_and_load_img("../Images/" + scan1 + "/RAW/mpr-1.nifti.hdr")
    img2 = resize_and_load_img("../Images/" + scan2 + "/RAW/mpr-1.nifti.hdr")
    
    pair = np.stack([img1, img2], axis=0)

    scans.append(pair)
    labels.append(label)

x = np.array(scans)
y = np.array(labels)

print("X's shape: " + str(x.shape))
print("y's shape: " + str(y.shape))
"""

'\nscans = []\nlabels = []\n\n\nfor scan1, scan2, label in pairs:\n\n    img1 = resize_and_load_img("../Images/" + scan1 + "/RAW/mpr-1.nifti.hdr")\n    img2 = resize_and_load_img("../Images/" + scan2 + "/RAW/mpr-1.nifti.hdr")\n\n    pair = np.stack([img1, img2], axis=0)\n\n    scans.append(pair)\n    labels.append(label)\n\nx = np.array(scans)\ny = np.array(labels)\n\nprint("X\'s shape: " + str(x.shape))\nprint("y\'s shape: " + str(y.shape))\n'

In [ ]:
def convnext_block(x, filters, kernel_size=7):
    # Lightweight ConvNeXt-style block for 3D
    shortcut = x
    x = layers.LayerNormalization()(x)
    x = layers.Conv3D(filters, kernel_size, padding='same')(x)  # Depthwise
    x = layers.LayerNormalization()(x)
    x = layers.Conv3D(filters, 1, padding='same')(x)  # Pointwise

    shortcut = layers.Conv3D(filters=x.shape[-1], kernel_size=1, padding='same')(shortcut)
    x = layers.Add()([shortcut, x])
    return x

def feature_extractor(input_shape):
    inputs = Input(input_shape)
    x = layers.Conv3D(16, 3, padding='same', activation='gelu')(inputs)
    x = convnext_block(x, 16)
    x = layers.MaxPooling3D(2)(x)
    x = convnext_block(x, 32)
    x = layers.MaxPooling3D(2)(x)
    x = convnext_block(x, 64)
    x = layers.GlobalAveragePooling3D()(x)
    model = Model(inputs, x, name="feature_extractor")
    return model

def build_model():
    input_1 = Input((128, 128, 128, 1))
    input_2 = Input((128, 128, 128, 1))
    
    extractor = feature_extractor((128, 128, 128, 1))
    
    emb_1 = extractor(input_1)
    emb_2 = extractor(input_2)
    
    merged = layers.Concatenate()([emb_1, emb_2])
    x = layers.Dense(128, activation='gelu')(merged)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='gelu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(3, activation='softmax')(x)
    
    model = Model([input_1, input_2], output)
    return model

model = build_model()
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 128, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 128, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_extractor   │ (None, 64)        │    975,216 │ input_layer[0][0… │
│ (Functional)        │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ feature_extracto… │
│ (Concatenate)       │                   │            │ feature_extracto… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     16,512 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 3)         │        195 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,000,179 (3.82 MB)

 Trainable params: 1,000,179 (3.82 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
"""
rand_indxs = list(range(x.shape[0]))
np.random.shuffle(rand_indxs)
x = x[rand_indxs]
y = y[rand_indxs]

test_offset = int(x.shape[0]*0.9)

x = x[:test_offset]
y = y[:test_offset]

x1 = x[:, 0]
x2 = x[:, 1]

x_test = x[test_offset:]
y_test = y[test_offset:]

x_test_1 = x[:, 0]
x_test_2 = x[:, 1]
"""

'\nrand_indxs = list(range(x.shape[0]))\nnp.random.shuffle(rand_indxs)\nx = x[rand_indxs]\ny = y[rand_indxs]\n\ntest_offset = int(x.shape[0]*0.9)\n\nx = x[:test_offset]\ny = y[:test_offset]\n\nx1 = x[:, 0]\nx2 = x[:, 1]\n\nx_test = x[test_offset:]\ny_test = y[test_offset:]\n\nx_test_1 = x[:, 0]\nx_test_2 = x[:, 1]\n'

In [ ]:
def img_generator(pairs):
    while True:
        for scan1, scan2, label in pairs:
            img1 = resize_and_load_img(scan1)
            img2 = resize_and_load_img(scan2)

            x1 = np.array(img1).astype(np.float32)
            x2 = np.array(img2).astype(np.float32)
            y = np.array([label]).astype(np.int32)

            x1 = np.expand_dims(x1, axis=0)
            x2 = np.expand_dims(x2, axis=0)
            y = np.expand_dims(y, axis=0)

            yield ((x1, x2), y)

def batch_generator(pairs, batch_size=2):
    while True:
        x1_batch = []
        x2_batch = []
        y_batch = []

        for scan1, scan2, label in pairs:
            img1 = resize_and_load_img(scan1)
            img2 = resize_and_load_img(scan2)

            x1 = np.array(img1).astype(np.float32)
            x2 = np.array(img2).astype(np.float32)
            y = np.array(label).astype(np.int32)  # Not wrapped in list

            x1_batch.append(x1)
            x2_batch.append(x2)
            y_batch.append(y)

            if len(x1_batch) == batch_size:
                # Stack into batch arrays
                yield ([np.stack(x1_batch), np.stack(x2_batch)],
                       np.array(y_batch))
                x1_batch, x2_batch, y_batch = [], [], []

pairs_shuffled = pairs[:]
random.shuffle(pairs_shuffled)

pairs_shuffled = np.array(pairs_shuffled)

val_offset = int(pairs_shuffled.shape[0]*0.7)
test_offset = int(pairs_shuffled.shape[0]*0.9)

train_gen = batch_generator(pairs_shuffled[:val_offset])
val_gen = batch_generator(pairs_shuffled[val_offset:test_offset])

# print("train_gen, next:" + str(next(train_gen)))
#################
sample = next(train_gen)

inputs, label = sample

print("Type of inputs:", type(inputs))
print("Type of inputs[0]:", type(inputs[0]))
print("Shape of inputs[0]:", inputs[0].shape)
print("Type of inputs[1]:", type(inputs[1]))
print("Shape of inputs[1]:", inputs[1].shape)

print("Type of label:", type(label))
print("Shape of label:", label.shape)
print("Label value:", label)

# Get one sample from your generator
(sample_x1, sample_x2), sample_y = next(train_gen)

# Ensure sample_x1 and sample_x2 are batched
print(sample_x1.shape == (1, 128, 128, 128, 1))
print(sample_x2.shape == (1, 128, 128, 128, 1))





[[-5.5543476e-01]
 [-5.0045890e-01]
 [-5.3980070e-01]
 [-3.8455015e-01]
 [-5.9685975e-01]
 [-4.9389371e-01]
 [-4.7560489e-01]
 [-5.9412211e-01]
 [-5.0193691e-01]
 [-4.1181457e-01]
 [-2.8395125e-01]
 [-5.2459961e-01]
 [-5.0680995e-01]
 [-5.5990344e-01]
 [-4.9802679e-01]
 [-5.0680053e-01]
 [ 5.0129008e-01]
 [ 9.0311491e-01]
 [ 8.4068006e-01]
 [ 7.9270303e-01]
 [ 1.2973431e+00]
 [ 1.6892248e+00]
 [ 2.0812120e+00]
 [ 6.0159999e-01]
 [ 5.6559187e-01]
 [ 1.3590527e+00]
 [ 1.9803665e+00]
 [ 1.6473038e+00]
 [ 6.2907416e-01]
 [ 3.7992457e-01]
 [ 5.9102905e-01]
 [ 5.1810569e-01]
 [ 4.9596575e-01]
 [ 5.9459430e-01]
 [ 7.8399163e-01]
 [ 1.1704428e+00]
 [ 2.1479604e+00]
 [ 2.4995527e+00]
 [ 2.4130218e+00]
 [ 4.4969863e-01]
 [ 8.2906276e-01]
 [ 1.2087959e+00]
 [ 9.1014665e-01]
 [ 1.0900655e+00]
 [ 9.3940681e-01]
 [ 4.2133480e-01]
 [ 1.3043668e+00]
 [ 2.2956045e+00]
 [ 2.6011491e+00]
 [ 2.8293067e-01]
 [ 1.7186137e-01]
 [-2.8618040e-02]
 [ 5.9927607e-01]
 [ 6.6540492e-01]
 [ 8.8745594e-01]
 [ 2.31083

In [ ]:
tensorboard_callback = TensorBoard(log_dir='logs', histogram_freq=1)
csv_logger = CSVLogger('training_log.csv', append=True)

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.000001)

history = model.fit(train_gen,
    steps_per_epoch=val_offset,
    validation_data=val_gen,
    validation_steps=test_offset-val_offset,
    epochs=50,
    callbacks=[early_stop, csv_logger]
)

Epoch 1/50
156/156 ━━━━━━━━━━━━━━━━━━━━ 822s 5s/step - accuracy: 0.8255 - loss: 0.5966 - val_accuracy: 0.7955 - val_loss: 0.6535
Epoch 2/50
156/156 ━━━━━━━━━━━━━━━━━━━━ 834s 5s/step - accuracy: 0.8225 - loss: 0.6638 - val_accuracy: 0.8182 - val_loss: 0.5860
Epoch 3/50
156/156 ━━━━━━━━━━━━━━━━━━━━ 830s 5s/step - accuracy: 0.8373 - loss: 0.5513 - val_accuracy: 0.7955 - val_loss: 0.6479
Epoch 4/50
156/156 ━━━━━━━━━━━━━━━━━━━━ 832s 5s/step - accuracy: 0.8313 - loss: 0.5342 - val_accuracy: 0.8182 - val_loss: 0.5535
Epoch 5/50
156/156 ━━━━━━━━━━━━━━━━━━━━ 798s 5s/step - accuracy: 0.8373 - loss: 0.5594 - val_accuracy: 0.8182 - val_loss: 0.5566
Epoch 6/50
156/156 ━━━━━━━━━━━━━━━━━━━━ 814s 5s/step - accuracy: 0.8328 - loss: 0.5720 - val_accuracy: 0.8182 - val_loss: 0.5552
Epoch 7/50
 15/156 ━━━━━━━━━━━━━━━━━━━━ 11:20 5s/step - accuracy: 0.7967 - loss: 0.8457

KeyboardInterrupt: 

In [ ]:
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.plot(history.history['accuracy'], label='accuracy')
plt.xlabel("Epoch")
plt.legend()
plt.grid(True)
plt.show()